# Regression metrics (R² and RMSE (Root Mean Squared Error))

_Machine Learning_

**How well does the line fit? Compare its errors to a baseline.**

For number predictions, we need scores for "how good is the fit".

     RMSE is the typical size of the errors, in the same units as $y$.

---

This notebook is a practice scaffold for the **Regression metrics (R² and RMSE (Root Mean Squared Error))** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, and the target is a continuous value.

In [ ]:
from sklearn.datasets import load_diabetes

data = load_diabetes(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target summary:")
print(data.target.describe())
data.frame.head()

## Reference implementation — scikit-learn

### Step 1 — Make data and split into train / test

We generate a synthetic regression problem: 400 examples, 5 informative features, and some random `noise` so no model can fit it perfectly. We then hold out 30% of the rows as a **test set** the model never sees during fitting — metrics are only meaningful on data the model didn't train on.

In [ ]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split

# 400 examples, 5 features, with measurement noise so the fit is imperfect.
X, y = make_regression(n_samples=400, n_features=5, noise=15.0, random_state=0)

# Hold out 30% as a test set the model never trains on.
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)

print("train rows:", Xtr.shape[0], " test rows:", Xte.shape[0])

### Step 2 — Fit a line and score it with RMSE and R²

`LinearRegression` finds the weights that minimise squared error on the training set. On the held-out test set we compute two scores. **RMSE** is the square root of the mean squared error — the typical error size, in the same units as `y`. **R²** is the fraction of the target's variance the model explains: 1.0 is perfect, 0.0 means no better than guessing the mean.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Fit the line on the training data, then predict the held-out test rows.
model = LinearRegression().fit(Xtr, ytr)
pred = model.predict(Xte)

# RMSE = sqrt(mean squared error); same units as y.
mse = mean_squared_error(yte, pred)
rmse = np.sqrt(mse)
print("RMSE:", round(rmse, 3), "(same units as y)")

# R^2: fraction of variance explained (1.0 = perfect).
r2 = r2_score(yte, pred)
print("R^2 :", round(r2, 4))

### Step 3 — Compare against the mean baseline

R² is defined relative to a dumb baseline: **always predict the training mean**. That baseline scores R² = 0 by construction. Seeing it makes the model's R² concrete — anything above 0 means the model beats just guessing the average.

In [ ]:
# Baseline: ignore the features and always predict the training mean.
baseline = np.full_like(yte, ytr.mean())

# By definition this scores R^2 = 0.
baseline_r2 = r2_score(yte, baseline)
print("baseline R^2:", round(baseline_r2, 4))

## Visualize the data & results

_How close are a linear model's diabetes-progression predictions to the true values?_

### Step 1 — Train a model on the real diabetes data

We switch from synthetic data to 442 real patients with all 10 features and a disease-progression target. Same recipe: split into train / test, then fit a linear model and predict the held-out patients.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# 442 real patients, all 10 features, progression target.
db = load_diabetes()
Xtr, Xte, ytr, yte = train_test_split(db.data, db.target, test_size=0.3, random_state=0)

model = LinearRegression().fit(Xtr, ytr)
pred = model.predict(Xte)

### Step 2 — Plot predicted vs actual

Each dot is one test patient: its true progression on the x-axis, the model's prediction on the y-axis. The dashed diagonal is **perfect prediction** — points hugging that line are accurate, and the vertical spread away from it is exactly the error RMSE summarises.

In [ ]:
# Each dot: true value (x) vs predicted value (y) for one test patient.
plt.scatter(yte, pred, c="#4ea1ff", edgecolor="k", label="test patients")

# Dashed diagonal = perfect prediction (predicted equals actual).
lo, hi = yte.min(), yte.max()
plt.plot([lo, hi], [lo, hi], "--", color="#ffb454", label="perfect prediction")

plt.xlabel("actual progression")
plt.ylabel("predicted progression")
plt.title("Predicted vs actual")
plt.legend()
plt.show()